# Experiment 2: Custom 3D U-Net Pipeline

This notebook implements a **custom 3D U-Net** for cell detection, trained on the ground-truth
`.geff` centroids from the training set. 

**Pipeline Overview**:
1. **Target Generation**: Convert `.geff` point centroids into 3D Gaussian heatmaps.
2. **Training**: Train a lightweight 3D U-Net to predict these heatmaps from raw volumes.
3. **Inference**: Predict heatmap → find local maxima → extract centroids.
4. **Evaluation**: Build tracking graph and compute competition Jaccard metrics.

> **Note:** Training 3D U-Nets requires a GPU for reasonable speed. This notebook is configured
> to run a small demonstration (few samples/epochs) by default.

## Imports

In [7]:
import json
import os
import time
import traceback

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

# Chunk decompressor fallback
try:
    from numcodecs import Blosc as _NumcodecsBlosc
    _USE_NUMCODECS = True
except ImportError:
    _USE_NUMCODECS = False
    import blosc2

PyTorch version: 2.9.0+cpu
Using device: cpu


## Configuration

In [8]:
DATA_DIR = "f:/4D cell tracking/biohub-cell-tracking-during-development"
SPLIT = "train"

VOXEL_SCALE = np.array([1.625, 0.40625, 0.40625])  # z, y, x

# Data splitting for this experiment
NUM_TRAIN_SAMPLES = 2    # Small number for demonstration
NUM_EVAL_SAMPLES = 1
FRAMES_PER_SAMPLE = 30   # Only train on first 10 frames per sample to save time

# U-Net & Heatmap Parameters
GAUSSIAN_SIGMA = 2.0     # Spread of the target heatmap spots
BATCH_SIZE = 1
EPOCHS = 10
LEARNING_RATE = 1e-3

# Inference Parameters
PEAK_MIN_DISTANCE = 5    # Minimum distance between detected centroids (voxels)
PEAK_THRESHOLD_ABS = 0.2 # Minimum heatmap value to be considered a cell

# Tracker parameters
MAX_LINK_DISTANCE_UM = 7.0

## Data I/O (Same as baseline)

In [9]:
def get_array_meta(zarr_path: str) -> dict:
    with open(os.path.join(zarr_path, "0", "zarr.json"), "r") as f:
        return json.load(f)

def load_volume_frame(zarr_path: str, t: int) -> np.ndarray:
    meta = get_array_meta(zarr_path)
    shape = tuple(meta["shape"])
    chunk_shape = tuple(meta["chunk_grid"]["configuration"]["chunk_shape"])
    dtype = np.dtype(meta["data_type"]).newbyteorder("<")
    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    with open(chunk_path, "rb") as f:
        raw = f.read()
    if _USE_NUMCODECS:
        codec = _NumcodecsBlosc(cname="zstd", clevel=1, shuffle=_NumcodecsBlosc.BITSHUFFLE)
        buf = codec.decode(raw)
    else:
        buf = blosc2.decompress(raw)
    return np.frombuffer(buf, dtype=dtype).reshape(chunk_shape)[0]

def voxel_to_physical(coords_zyx: np.ndarray) -> np.ndarray:
    return coords_zyx * VOXEL_SCALE

def load_geff(geff_path: str) -> dict:
    from numcodecs import Zstd
    meta_file = os.path.join(geff_path, "zarr.json")
    with open(meta_file, "r") as f:
        meta = json.load(f)

    def read_arr(path, dtype):
        zstd = Zstd(level=0)
        cf = os.path.join(path, "c", "0")
        if not os.path.isfile(cf):
            cf = os.path.join(path, "c", "0", "0")
        with open(cf, "rb") as f:
            raw = f.read()
        return np.frombuffer(zstd.decode(raw), dtype=dtype)

    node_ids = read_arr(os.path.join(geff_path, "nodes", "ids"), "<u8")
    props = {}
    for prop in ["t", "z", "y", "x"]:
        props[prop] = read_arr(os.path.join(geff_path, "nodes", "props", prop, "values"), "<i8")
    nodes_df = pd.DataFrame({"node_id": node_ids, **props})
    
    edges_path = os.path.join(geff_path, "edges", "ids")
    with open(os.path.join(edges_path, "zarr.json"), "r") as f:
        n_edges = json.load(f)["shape"][0]
    edges = read_arr(edges_path, "<u8").reshape(n_edges, 2)

    return {"node_ids": node_ids, "nodes_df": nodes_df, "edges": edges, "metadata": meta}

def list_samples(data_dir: str, split: str) -> list:
    split_dir = os.path.join(data_dir, split)
    return sorted(set(e.replace(".zarr", "") for e in os.listdir(split_dir) if e.endswith(".zarr")))

## 1. Target Heatmap Generation
We train the network to predict a Gaussian blob at each cell centroid.

In [10]:
def generate_heatmap(shape, centroids, sigma=GAUSSIAN_SIGMA):
    """Creates a 3D volume of `shape` with Gaussians at `centroids`."""
    heatmap = np.zeros(shape, dtype=np.float32)
    if len(centroids) == 0:
        return heatmap
        
    # Create a small 3D gaussian kernel
    radius = int(sigma * 3)
    z, y, x = np.ogrid[-radius:radius+1, -radius:radius+1, -radius:radius+1]
    kernel = np.exp(-(x**2 + y**2 + z**2) / (2 * sigma**2))
    
    Z, Y, X = shape
    for c in centroids:
        cz, cy, cx = int(round(c[0])), int(round(c[1])), int(round(c[2]))
        
        # Calculate bounding box for the kernel
        z_min, z_max = max(0, cz - radius), min(Z, cz + radius + 1)
        y_min, y_max = max(0, cy - radius), min(Y, cy + radius + 1)
        x_min, x_max = max(0, cx - radius), min(X, cx + radius + 1)
        
        # Calculate indices within the kernel
        kz_min = z_min - (cz - radius)
        kz_max = kz_min + (z_max - z_min)
        ky_min = y_min - (cy - radius)
        ky_max = ky_min + (y_max - y_min)
        kx_min = x_min - (cx - radius)
        kx_max = kx_min + (x_max - x_min)
        
        if kz_min >= kz_max or ky_min >= ky_max or kx_min >= kx_max:
            continue
            
        # Add kernel to heatmap (taking max to handle overlapping cells gracefully)
        heatmap[z_min:z_max, y_min:y_max, x_min:x_max] = np.maximum(
            heatmap[z_min:z_max, y_min:y_max, x_min:x_max],
            kernel[kz_min:kz_max, ky_min:ky_max, kx_min:kx_max]
        )
        
    return heatmap

## 2. 3D U-Net Architecture
A lightweight fully convolutional 3D U-Net.

In [11]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[16, 32, 64]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

        # Encoder
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose3d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv3d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            
            # Handle potential shape mismatch due to pooling
            if x.shape != skip_connection.shape:
                x = F.interpolate(x, size=skip_connection.shape[2:], mode="trilinear", align_corners=False)
                
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](concat_skip)

        return torch.sigmoid(self.final_conv(x))

## 3. Dataset & Training Loop

In [12]:
class CellTrackingDataset(Dataset):
    def __init__(self, samples, data_dir, split="train", max_frames=FRAMES_PER_SAMPLE):
        self.items = [] # list of (zarr_path, geff_path, t)
        for sample in samples:
            zarr_p = os.path.join(data_dir, split, f"{sample}.zarr")
            geff_p = os.path.join(data_dir, split, f"{sample}.geff")
            meta = get_array_meta(zarr_p)
            T = min(meta["shape"][0], max_frames)
            for t in range(T):
                self.items.append((zarr_p, geff_p, t))
        
        print(f"Dataset built: {len(self.items)} frames from {len(samples)} samples.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        zarr_p, geff_p, t = self.items[idx]
        
        # Load volume
        vol = load_volume_frame(zarr_p, t).astype(np.float32)
        # Normalize volume to 0-1 (using 1st and 99.9th percentile)
        p_low, p_high = np.percentile(vol, [1, 99.9])
        vol = np.clip((vol - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
        # Load GT centroids
        gt = load_geff(geff_p)
        df = gt["nodes_df"]
        frame_nodes = df[df["t"] == t]
        centroids = frame_nodes[["z", "y", "x"]].values
        
        # Generate heatmap target
        heatmap = generate_heatmap(vol.shape, centroids)
        
        # Add channel dimension
        vol = torch.from_numpy(vol).float().unsqueeze(0)
        heatmap = torch.from_numpy(heatmap).float().unsqueeze(0)
        
        return vol, heatmap

In [13]:
# Get samples
all_samples = list_samples(DATA_DIR, SPLIT)
train_samples = all_samples[:NUM_TRAIN_SAMPLES]
eval_samples = all_samples[NUM_TRAIN_SAMPLES:NUM_TRAIN_SAMPLES+NUM_EVAL_SAMPLES]

print(f"Train samples: {train_samples}")
print(f"Eval samples: {eval_samples}")

# Create dataloader
train_dataset = CellTrackingDataset(train_samples, DATA_DIR, SPLIT)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model
model = UNet3D(in_channels=1, out_channels=1).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    t0 = time.time()
    
    for i, (vols, targets) in enumerate(train_loader):
        vols, targets = vols.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad()
        preds = model(vols)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} Time: {time.time()-t0:.1f}s")

print("Training complete.")

Train samples: ['44b6_0113de3b', '44b6_0b24845f']
Eval samples: ['44b6_0c582fdc']
Dataset built: 60 frames from 2 samples.

Starting training...
Epoch [1/10] Loss: 0.0877 Time: 352.7s
Epoch [2/10] Loss: 0.0335 Time: 350.8s
Epoch [3/10] Loss: 0.0165 Time: 364.2s
Epoch [4/10] Loss: 0.0099 Time: 426.4s
Epoch [5/10] Loss: 0.0065 Time: 423.5s
Epoch [6/10] Loss: 0.0044 Time: 424.3s
Epoch [7/10] Loss: 0.0033 Time: 420.8s
Epoch [8/10] Loss: 0.0025 Time: 413.5s
Epoch [9/10] Loss: 0.0020 Time: 418.2s
Epoch [10/10] Loss: 0.0016 Time: 422.8s
Training complete.


## 4. Inference & Tracker integration

In [14]:
def detect_cells_unet(volume_3d: np.ndarray, voxel_scale=None) -> np.ndarray:
    """Predict heatmap and extract peaks as centroids."""
    model.eval()
    with torch.no_grad():
        # Preprocess
        vol = volume_3d.astype(np.float32)
        p_low, p_high = np.percentile(vol, [1, 99.9])
        vol = np.clip((vol - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
        t_vol = torch.from_numpy(vol).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        
        # Predict
        pred_heatmap = model(t_vol).squeeze().cpu().numpy()
        
    # Extract peaks (local maxima)
    centroids = peak_local_max(
        pred_heatmap, 
        min_distance=PEAK_MIN_DISTANCE, 
        threshold_abs=PEAK_THRESHOLD_ABS
    )
    
    return centroids.astype(np.float64)

In [15]:
# Copy-paste tracker & metric logic from baseline for self-containment
class TrackGraph:
    def __init__(self):
        self.nodes = []
        self.edges = []
        self._next_id = 1
    def add_node(self, t, z, y, x):
        nid = self._next_id; self._next_id += 1
        self.nodes.append({"id": nid, "t": t, "z": int(round(z)), "y": int(round(y)), "x": int(round(x))})
        return nid
    def add_edge(self, src, tgt): self.edges.append((src, tgt))
    def get_nodes_at_time(self, t): return [n for n in self.nodes if n["t"] == t]
    def get_outgoing_edges(self, nid): return [tgt for src, tgt in self.edges if src == nid]
    def get_incoming_edges(self, nid): return [src for src, tgt in self.edges if tgt == nid]

def link_frames(c0, c1, max_dist=MAX_LINK_DISTANCE_UM):
    if len(c0) == 0 or len(c1) == 0: return []
    cost = cdist(c0 * VOXEL_SCALE, c1 * VOXEL_SCALE)
    cost_m = np.where(cost <= max_dist, cost, 1e6)
    ri, ci = linear_sum_assignment(cost_m)
    return [(int(i), int(j)) for i, j in zip(ri, ci) if cost[i, j] <= max_dist]

def build_tracks(all_centroids, max_dist=MAX_LINK_DISTANCE_UM):
    graph = TrackGraph()
    timepoints = sorted(all_centroids.keys())
    if not timepoints: return graph
    frame_ids = {}
    for t in timepoints:
        frame_ids[t] = [graph.add_node(t, *c) for c in all_centroids[t]]
    for i in range(len(timepoints) - 1):
        t0, t1 = timepoints[i], timepoints[i+1]
        if t1 != t0 + 1: continue
        for i0, i1 in link_frames(all_centroids[t0], all_centroids[t1], max_dist):
            graph.add_edge(frame_ids[t0][i0], frame_ids[t1][i1])
    return graph

def detect_divisions(graph, all_centroids, max_dist=MAX_LINK_DISTANCE_UM):
    timepoints = sorted(all_centroids.keys())
    for i in range(1, len(timepoints)):
        t, tp = timepoints[i], timepoints[i-1]
        if t != tp + 1: continue
        nodes_t = graph.get_nodes_at_time(t)
        unmatched = [n for n in nodes_t if not graph.get_incoming_edges(n["id"])]
        if not unmatched: continue
        nodes_prev = graph.get_nodes_at_time(tp)
        if not nodes_prev: continue
        p_phys = np.array([[n["z"], n["y"], n["x"]] for n in nodes_prev], dtype=float) * VOXEL_SCALE
        p_ids = [n["id"] for n in nodes_prev]
        for node in unmatched:
            c_phys = np.array([[node["z"], node["y"], node["x"]]], dtype=float) * VOXEL_SCALE
            dists = cdist(c_phys, p_phys)[0]
            idx = np.argmin(dists)
            if dists[idx] <= max_dist and len(graph.get_outgoing_edges(p_ids[idx])) == 1:
                graph.add_edge(p_ids[idx], node["id"])
    return graph

def run_pipeline(zarr_path, detector_fn, verbose=True):
    meta = get_array_meta(zarr_path)
    T = min(meta["shape"][0], FRAMES_PER_SAMPLE) # only evaluate first N frames for speed
    all_centroids = {}
    for t in range(T):
        frame = load_volume_frame(zarr_path, t)
        centroids = detector_fn(frame)
        all_centroids[t] = centroids
        if verbose and (t % 10 == 0 or t == T - 1):
            print(f"    t={t:3d}: {len(centroids):4d} detections")
    graph = build_tracks(all_centroids)
    return detect_divisions(graph, all_centroids)

In [16]:
def _match_nodes_at_time(pred_nodes, gt_nodes, max_dist_um=7.0):
    if not pred_nodes or not gt_nodes: return {}
    pc = np.array([[n["z"], n["y"], n["x"]] for n in pred_nodes], dtype=float)
    gc = np.array([[n["z"], n["y"], n["x"]] for n in gt_nodes], dtype=float)
    cost = cdist(pc * VOXEL_SCALE, gc * VOXEL_SCALE)
    cost_m = np.where(cost <= max_dist_um, cost, 1e6)
    ri, ci = linear_sum_assignment(cost_m)
    return {pred_nodes[i]["id"]: gt_nodes[j]["node_id"] for i, j in zip(ri, ci) if cost[i, j] <= max_dist_um}

def evaluate_sample_partial(pred_graph, gt_geff, frames=FRAMES_PER_SAMPLE, max_dist_um=7.0):
    """Evaluates only up to `frames`."""
    gt_df = gt_geff["nodes_df"]
    gt_df = gt_df[gt_df["t"] < frames]
    gt_edges_arr = gt_geff["edges"]
    
    gt_nodes = set(gt_df["node_id"])
    gt_edges = [e for e in gt_edges_arr if e[0] in gt_nodes and e[1] in gt_nodes]

    gt_by_t = {}
    for _, row in gt_df.iterrows():
        gt_by_t.setdefault(int(row["t"]), []).append(
            {"node_id": int(row["node_id"]), "z": int(row["z"]), "y": int(row["y"]), "x": int(row["x"])})

    pred_by_t = {}
    for n in pred_graph.nodes:
        pred_by_t.setdefault(n["t"], []).append(n)

    pred_to_gt = {}
    for t in sorted(set(list(gt_by_t) + list(pred_by_t))):
        pred_to_gt.update(_match_nodes_at_time(pred_by_t.get(t, []), gt_by_t.get(t, []), max_dist_um))

    gt_edge_set = {(int(e[0]), int(e[1])) for e in gt_edges}
    pred_mapped = set()
    for s, tg in pred_graph.edges:
        gs, gtg = pred_to_gt.get(s), pred_to_gt.get(tg)
        if gs is not None and gtg is not None: pred_mapped.add((gs, gtg))

    etp = len(pred_mapped & gt_edge_set)
    efp = len(pred_mapped - gt_edge_set)
    efn = len(gt_edge_set - pred_mapped)
    ej = etp / (etp + efp + efn) if (etp + efp + efn) > 0 else 0.0

    return {"edge_jaccard": ej, "edge_tp": etp, "edge_fp": efp, "edge_fn": efn, "n_pred": len(pred_graph.nodes)}

## 5. Evaluation Loop

In [17]:
print("\nEvaluating on Eval Samples...")
for sample in eval_samples:
    zarr_p = os.path.join(DATA_DIR, SPLIT, f"{sample}.zarr")
    geff_p = os.path.join(DATA_DIR, SPLIT, f"{sample}.geff")
    gt = load_geff(geff_p)
    
    print(f"\nSample: {sample}")
    t0 = time.time()
    graph = run_pipeline(zarr_p, detect_cells_unet)
    elapsed = time.time() - t0
    print(f"  Inference time (first {FRAMES_PER_SAMPLE} frames): {elapsed:.1f}s")
    
    res = evaluate_sample_partial(graph, gt)
    print(f"  Edge Jaccard: {res['edge_jaccard']:.4f} (TP={res['edge_tp']} FP={res['edge_fp']} FN={res['edge_fn']})")
    print(f"  Predicted nodes: {res['n_pred']}")


Evaluating on Eval Samples...

Sample: 44b6_0c582fdc
    t=  0:    0 detections
    t= 10:    0 detections
    t= 20:    0 detections
    t= 29:    0 detections
  Inference time (first 30 frames): 87.0s
  Edge Jaccard: 0.0000 (TP=0 FP=0 FN=9)
  Predicted nodes: 1
